In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔍 Search Engine Optimization & Semantic Search Guide

**Optimizing for search engines and implementing full-text semantic search capabilities**

This guide covers:
- SEO fundamentals (meta tags, structured data, sitemaps)
- Search engine indexing
- Semantic search with embeddings
- Full-text search engines (Elasticsearch, Algolia)
- Search ranking algorithms
- Query optimization
- Analytics and monitoring
- Performance tuning
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. SEO Fundamentals

### Meta Tags & Structured Data
```html
<!-- index.html - SEO-optimized page -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    
    <!-- Essential Meta Tags -->
    <title>Interactive AI Character Platform | Aria</title>
    <meta name="description" content="Build interactive AI characters with natural language commands, real-time animations, and multi-provider chat backends.">
    <meta name="keywords" content="AI, character, interactive, quantum ML, chat">
    
    <!-- Open Graph (Social Media) -->
    <meta property="og:type" content="website">
    <meta property="og:title" content="Aria - Interactive AI Character Platform">
    <meta property="og:description" content="Build and interact with intelligent AI characters">
    <meta property="og:image" content="https://aria.example.com/og-image.png">
    <meta property="og:url" content="https://aria.example.com">
    
    <!-- Twitter Cards -->
    <meta name="twitter:card" content="summary_large_image">
    <meta name="twitter:title" content="Aria Platform">
    <meta name="twitter:description" content="Interactive AI characters with real-time interactions">
    <meta name="twitter:image" content="https://aria.example.com/twitter-image.png">
    
    <!-- Canonical URL -->
    <link rel="canonical" href="https://aria.example.com">
    
    <!-- Structured Data (Schema.org JSON-LD) -->
    <script type="application/ld+json">
    {
        "@context": "https://schema.org",
        "@type": "SoftwareApplication",
        "name": "Aria",
        "description": "Interactive AI character platform",
        "url": "https://aria.example.com",
        "applicationCategory": "Multimedia",
        "offers": {
            "@type": "Offer",
            "price": "0",
            "priceCurrency": "USD"
        },
        "aggregateRating": {
            "@type": "AggregateRating",
            "ratingValue": "4.8",
            "ratingCount": "150"
        }
    }
    </script>
</head>
<body>
    <!-- Content -->
</body>
</html>
```

### Sitemap & Robots.txt
```xml
<!-- sitemap.xml -->
<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
    <url>
        <loc>https://aria.example.com</loc>
        <lastmod>2026-07-05</lastmod>
        <changefreq>weekly</changefreq>
        <priority>1.0</priority>
    </url>
    <url>
        <loc>https://aria.example.com/docs</loc>
        <lastmod>2026-07-05</lastmod>
        <changefreq>weekly</changefreq>
        <priority>0.9</priority>
    </url>
    <url>
        <loc>https://aria.example.com/api</loc>
        <lastmod>2026-07-05</lastmod>
        <changefreq>monthly</changefreq>
        <priority>0.8</priority>
    </url>
</urlset>
```

```
# robots.txt
User-agent: *
Allow: /
Allow: /api/
Disallow: /admin
Disallow: /private

Sitemap: https://aria.example.com/sitemap.xml
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Full-Text Search Engines

### Elasticsearch Implementation
```python
from elasticsearch import Elasticsearch
from datetime import datetime

class ElasticsearchIndex:
    def __init__(self, host: str = "localhost:9200"):
        self.es = Elasticsearch([host])
        self.index_name = "aria-docs"
    
    def create_index(self):
        """Create index with custom analyzer"""
        settings = {
            "settings": {
                "number_of_shards": 3,
                "number_of_replicas": 1,
                "analysis": {
                    "analyzer": {
                        "custom_analyzer": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": [
                                "lowercase",
                                "stop",
                                "snowball"
                            ]
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "title": {
                        "type": "text",
                        "analyzer": "custom_analyzer",
                        "fields": {
                            "keyword": {"type": "keyword"}
                        }
                    },
                    "content": {
                        "type": "text",
                        "analyzer": "custom_analyzer"
                    },
                    "url": {"type": "keyword"},
                    "created_at": {"type": "date"},
                    "category": {"type": "keyword"},
                    "embedding": {
                        "type": "dense_vector",
                        "dims": 384,
                        "index": True,
                        "similarity": "cosine"
                    }
                }
            }
        }
        
        if self.es.indices.exists(index=self.index_name):
            self.es.indices.delete(index=self.index_name)
        
        self.es.indices.create(index=self.index_name, body=settings)
    
    def index_document(self, doc_id: str, title: str, content: str, embedding: list):
        """Index a document"""
        doc = {
            "title": title,
            "content": content,
            "url": f"/docs/{doc_id}",
            "created_at": datetime.now(),
            "category": "documentation",
            "embedding": embedding
        }
        
        self.es.index(index=self.index_name, id=doc_id, body=doc)
    
    def search(self, query: str, size: int = 10):
        """Full-text search"""
        search_body = {
            "query": {
                "multi_match": {
                    "query": query,
                    "fields": ["title^2", "content"]
                }
            },
            "size": size
        }
        
        results = self.es.search(index=self.index_name, body=search_body)
        return results['hits']['hits']
    
    def semantic_search(self, embedding: list, size: int = 10):
        """Search by embedding similarity"""
        search_body = {
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.query_embedding, 'embedding') + 1.0",
                        "params": {
                            "query_embedding": embedding
                        }
                    }
                }
            },
            "size": size
        }
        
        results = self.es.search(index=self.index_name, body=search_body)
        return results['hits']['hits']

# Usage
es_index = ElasticsearchIndex()
es_index.create_index()

# Index documents
es_index.index_document(
    "doc-1",
    "Aria Character System",
    "Learn how to create interactive AI characters...",
    [0.1, 0.2, 0.3, ...]  # embedding vector
)

# Search
results = es_index.search("AI character creation")
```

### Algolia for Frontend Search
```python
import algoliasearch

class AlgoliaSearchClient:
    def __init__(self, app_id: str, admin_key: str):
        self.client = algoliasearch.SearchClient.create(app_id, admin_key)
        self.index = self.client.init_index("aria_docs")
    
    def index_documents(self, documents: list):
        """Batch index documents"""
        self.index.save_objects(documents)
    
    def search(self, query: str):
        """Search with typo tolerance"""
        results = self.index.search(query, {
            "typoTolerance": True,
            "hitsPerPage": 10,
            "attributesToHighlight": ["title", "content"]
        })
        return results

# Frontend JavaScript
# const search = instantsearch({
#     indexName: 'aria_docs',
#     searchClient: algoliasearch('APP_ID', 'SEARCH_KEY')
# });
#
# search.addWidget(
#     instantsearch.widgets.searchBox({
#         container: '#searchbox'
#     })
# );
#
# search.addWidget(
#     instantsearch.widgets.hits({
#         container: '#results'
#     })
# );
#
# search.start();
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Semantic Search with Embeddings

### Embedding Generation & Storage
```python
from sentence_transformers import SentenceTransformer
import numpy as np

class SemanticSearch:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    
    def embed_text(self, text: str) -> np.ndarray:
        """Generate embedding for text"""
        embedding = self.model.encode(text, convert_to_numpy=True)
        return embedding
    
    def embed_documents(self, texts: list) -> np.ndarray:
        """Generate embeddings for multiple texts"""
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings
    
    def semantic_search(self, query: str, documents: list, top_k: int = 5):
        """Find similar documents"""
        query_embedding = self.embed_text(query)
        doc_embeddings = self.embed_documents(documents)
        
        # Calculate cosine similarity
        similarities = np.dot(doc_embeddings, query_embedding) / (
            np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        
        # Get top k
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = [
            {
                "document": documents[idx],
                "similarity": float(similarities[idx])
            }
            for idx in top_indices
        ]
        
        return results

# Usage
semantic_search = SemanticSearch()

documents = [
    "How to create Aria character commands",
    "API gateway rate limiting strategies",
    "Database sharding techniques"
]

results = semantic_search.semantic_search(
    "character interaction",
    documents,
    top_k=3
)

for result in results:
    print(f"Match: {result['document']} (similarity: {result['similarity']:.4f})")
```

### Vector Database Integration
```python
from pinecone import Pinecone

class VectorStore:
    def __init__(self, api_key: str, index_name: str = "aria-embeddings"):
        self.pc = Pinecone(api_key=api_key)
        self.index = self.pc.Index(index_name)
    
    def add_vectors(self, vectors: list, metadata: list):
        """Add vectors to index"""
        vectors_to_insert = [
            (str(i), vec, meta)
            for i, (vec, meta) in enumerate(zip(vectors, metadata))
        ]
        self.index.upsert(vectors=vectors_to_insert)
    
    def search_similar(self, query_vector: list, top_k: int = 5):
        """Find similar vectors"""
        results = self.index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True
        )
        return results['matches']
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Search Optimization & Ranking

### Query Optimization
```python
class SearchOptimizer:
    """Optimize search queries and results"""
    
    @staticmethod
    def normalize_query(query: str) -> str:
        """Normalize search query"""
        # Remove extra whitespace
        query = ' '.join(query.split())
        # Convert to lowercase
        query = query.lower()
        # Remove special characters except quotes
        query = ''.join(c for c in query if c.isalnum() or c.isspace() or c in '"\'')
        return query
    
    @staticmethod
    def expand_query(query: str, synonyms: dict) -> list:
        """Expand query with synonyms"""
        expanded = [query]
        
        for word in query.split():
            if word in synonyms:
                expanded.extend(synonyms[word])
        
        return expanded
    
    @staticmethod
    def rank_results(results: list, boosts: dict) -> list:
        """Re-rank search results"""
        for result in results:
            result['score'] = result.get('score', 0)
            
            # Apply boosts
            for boost_key, boost_value in boosts.items():
                if boost_key in result.get('_source', {}):
                    result['score'] += boost_value
        
        return sorted(results, key=lambda x: x['score'], reverse=True)

# Usage
optimizer = SearchOptimizer()
normalized = optimizer.normalize_query("  Aria  Character   ")
```

### Performance Metrics
```python
class SearchMetrics:
    def __init__(self):
        self.metrics = []
    
    def track_search(self, query: str, results_count: int, latency_ms: float):
        """Track search metrics"""
        self.metrics.append({
            "query": query,
            "results": results_count,
            "latency": latency_ms
        })
    
    def get_p99_latency(self) -> float:
        """Get 99th percentile latency"""
        latencies = sorted([m['latency'] for m in self.metrics])
        return latencies[int(len(latencies) * 0.99)]
    
    def get_average_results(self) -> float:
        """Get average result count"""
        if not self.metrics:
            return 0
        return sum(m['results'] for m in self.metrics) / len(self.metrics)
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. Search Implementation Checklist

✅ **SEO**
- [ ] Meta tags and descriptions
- [ ] Structured data (Schema.org)
- [ ] Sitemap generated
- [ ] robots.txt configured
- [ ] Canonical URLs set
- [ ] Open Graph tags added

✅ **Full-Text Search**
- [ ] Search index created
- [ ] Analyzer configured
- [ ] Documents indexed
- [ ] Faceting configured
- [ ] Autocomplete enabled
- [ ] Highlighting working

✅ **Semantic Search**
- [ ] Embedding model selected
- [ ] Vectors generated
- [ ] Vector store provisioned
- [ ] Similarity search working
- [ ] Performance acceptable

✅ **Performance**
- [ ] Search latency < 200ms
- [ ] Queries cached
- [ ] Indexing optimized
- [ ] Pagination implemented
- [ ] Monitoring configured

✅ **Analytics**
- [ ] Popular searches tracked
- [ ] Zero-result queries identified
- [ ] Click-through rates measured
- [ ] Refinement funnel analyzed
</MySCode.Cell>
```